# Entrega 3 — Modelo Final, Interpretación y Comunicación
## Segmentación Semántica de Imágenes de Mascotas

**Curso:** Aprendizaje de Máquina Aplicado · EAFIT  
**Profesor:** Marco Terán  
**Autores:** Jose Blanco · Miguel Quijano  

---

### Objetivo de esta entrega

Entregar la versión final del proyecto respondiendo:

1. ¿Cuál es el mejor modelo y por qué?
2. ¿Qué tan confiables son sus resultados?
3. ¿Qué variables o patrones explican el desempeño?
4. ¿Qué conclusiones útiles deja el proyecto?
5. ¿Qué haría falta para mejorar o desplegar la solución?

### Mejoras sobre Entrega 2

| Aspecto | Entrega 2 | Entrega 3 |
|---------|-----------|----------|
| Resolución | 128×128 | **256×256** |
| Augmentación | Solo flip horizontal | **Flip + rotación + color jitter** |
| Loss | BCE | **BCE + Dice combinada** |
| Scheduler | Ninguno | **CosineAnnealingLR** |
| Épocas | 15 | **30 con early stopping** |
| Confiabilidad | 1 semilla | **3 semillas, media ± std** |
| Interpretabilidad | Ninguna | **Grad-CAM** |
| Análisis de sensibilidad | Ninguno | **Robustez ante perturbaciones** |

---
## 0. Setup del entorno

Este notebook está diseñado para **Google Colab con GPU T4**.

**Runtime → Change runtime type → T4 GPU**  
Tiempo estimado: ~25 min en T4.

In [ ]:
# 0.1 Instalar dependencias
!pip install -q segmentation-models-pytorch==0.3.4 tqdm

In [ ]:
# 0.2 Imports
import os
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import OxfordIIITPet
from torchvision import transforms as T

import segmentation_models_pytorch as smp
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# 0.3 Configuración
SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 30
PATIENCE = 7
LR = 1e-3
SEEDS = [42, 123, 7]

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 0.4 Crear directorios y descargar dataset
os.makedirs('../figures', exist_ok=True)
DATA_ROOT = '../data'

raw_trainval = OxfordIIITPet(
    root=DATA_ROOT, split='trainval',
    target_types='segmentation', download=True
)
raw_test = OxfordIIITPet(
    root=DATA_ROOT, split='test',
    target_types='segmentation', download=True
)
print(f'Trainval: {len(raw_trainval)}, Test: {len(raw_test)}')

---
## 1. Pipeline de datos mejorado

Respecto a Entrega 2:
- **Resolución 256×256** para capturar más detalle en bordes
- **Augmentación agresiva:** rotación (±15°), color jitter
- **Mismo split estratificado** (semilla 42) para comparabilidad

In [ ]:
# 1.1 Split estratificado por raza (idéntico a Entrega 2)
def get_breed(filename):
    return '_'.join(Path(filename).stem.split('_')[:-1])

trainval_breeds = [get_breed(p) for p in raw_trainval._images]

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(np.zeros(len(trainval_breeds)), trainval_breeds))
train_idx, val_idx = train_idx.tolist(), val_idx.tolist()

assert len(set(train_idx) & set(val_idx)) == 0
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(raw_test)}')

In [ ]:
# 1.2 Dataset con augmentación mejorada
class PetSegDataset(Dataset):
    def __init__(self, base_dataset, indices=None, img_size=IMG_SIZE,
                 augment=False, mean=None, std=None):
        self.base = base_dataset
        self.indices = indices if indices is not None else list(range(len(base_dataset)))
        self.img_size = img_size
        self.augment = augment
        self.mean = mean
        self.std = std
        self.to_tensor = T.ToTensor()
        self.color_jitter = T.ColorJitter(brightness=0.2, contrast=0.2,
                                          saturation=0.2, hue=0.05)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        img, mask = self.base[idx]

        img = T.functional.resize(img, (self.img_size, self.img_size))
        mask = T.functional.resize(mask, (self.img_size, self.img_size),
                                   interpolation=T.InterpolationMode.NEAREST)

        if self.augment:
            if torch.rand(1).item() > 0.5:
                img = T.functional.hflip(img)
                mask = T.functional.hflip(mask)
            if torch.rand(1).item() > 0.5:
                angle = random.uniform(-15, 15)
                img = T.functional.rotate(img, angle)
                mask = T.functional.rotate(mask, angle, fill=2)
            if torch.rand(1).item() > 0.5:
                img = self.color_jitter(img)

        img_t = self.to_tensor(img)
        if self.mean is not None and self.std is not None:
            img_t = T.functional.normalize(img_t, self.mean, self.std)

        mask_np = np.array(mask)
        binary_mask = (mask_np == 1).astype(np.float32)
        mask_t = torch.from_numpy(binary_mask).unsqueeze(0)
        return img_t, mask_t

In [ ]:
# 1.3 Stats de normalización SOLO sobre train (evita data leakage)
print('Calculando media/std sobre train (256x256)...')
temp_ds = PetSegDataset(raw_trainval, indices=train_idx, augment=False)
temp_loader = DataLoader(temp_ds, batch_size=64, shuffle=False, num_workers=2)

mean_acc, std_acc, n_samples = torch.zeros(3), torch.zeros(3), 0
for imgs, _ in tqdm(temp_loader, desc='Stats'):
    b = imgs.size(0)
    imgs_flat = imgs.view(b, 3, -1)
    mean_acc += imgs_flat.mean(dim=2).sum(dim=0)
    std_acc += imgs_flat.std(dim=2).sum(dim=0)
    n_samples += b

TRAIN_MEAN = (mean_acc / n_samples).tolist()
TRAIN_STD = (std_acc / n_samples).tolist()
print(f'Mean: {[f"{m:.4f}" for m in TRAIN_MEAN]}')
print(f'Std:  {[f"{s:.4f}" for s in TRAIN_STD]}')
del temp_ds, temp_loader

In [ ]:
# 1.4 Crear dataloaders
NUM_WORKERS = 2 if DEVICE.type == 'cuda' else 0

def create_loaders(seed=SEED):
    set_seed(seed)
    train_ds = PetSegDataset(raw_trainval, indices=train_idx, augment=True,
                             mean=TRAIN_MEAN, std=TRAIN_STD)
    val_ds = PetSegDataset(raw_trainval, indices=val_idx, augment=False,
                           mean=TRAIN_MEAN, std=TRAIN_STD)
    test_ds = PetSegDataset(raw_test, indices=None, augment=False,
                            mean=TRAIN_MEAN, std=TRAIN_STD)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = create_loaders()
imgs, masks = next(iter(train_loader))
print(f'Batch: imgs {tuple(imgs.shape)}, masks {tuple(masks.shape)}')
print(f'Imgs range: [{imgs.min():.2f}, {imgs.max():.2f}]')

---
## 2. Métricas y entrenamiento

In [ ]:
# 2.1 Métricas
def iou_score(preds, targets, threshold=0.5, eps=1e-7):
    preds_bin = (preds > threshold).float()
    intersection = (preds_bin * targets).sum(dim=(1, 2, 3))
    union = preds_bin.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - intersection
    return ((intersection + eps) / (union + eps)).mean().item()

def dice_score(preds, targets, threshold=0.5, eps=1e-7):
    preds_bin = (preds > threshold).float()
    intersection = (preds_bin * targets).sum(dim=(1, 2, 3))
    return ((2 * intersection + eps) /
            (preds_bin.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps)).mean().item()

# 2.2 Loss combinada: BCE + Dice
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, eps=1e-7):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCELoss()
        self.eps = eps

    def forward(self, preds, targets):
        bce_loss = self.bce(preds, targets)
        intersection = (preds * targets).sum(dim=(1, 2, 3))
        denom = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice_loss = 1 - ((2 * intersection + self.eps) / (denom + self.eps)).mean()
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

In [ ]:
# 2.3 Loop de entrenamiento con early stopping y scheduler
@torch.no_grad()
def evaluate_model(model, loader, criterion, threshold=0.5):
    model.eval()
    total_loss, total_iou, total_dice, n = 0, 0, 0, 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = torch.sigmoid(model(imgs))
        loss = criterion(preds, masks)
        b = imgs.size(0)
        total_loss += loss.item() * b
        total_iou += iou_score(preds, masks, threshold) * b
        total_dice += dice_score(preds, masks, threshold) * b
        n += b
    return {'loss': total_loss/n, 'iou': total_iou/n, 'dice': total_dice/n}


def train_model(model, train_loader, val_loader, criterion, epochs=EPOCHS,
                lr=LR, patience=PATIENCE, name='model'):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_dice': [], 'lr': []}
    best_val_iou = -1
    best_state = None
    no_improve = 0
    start = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss, n_batches = 0, 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            preds = torch.sigmoid(model(imgs))
            loss = criterion(preds, masks)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches += 1

        scheduler.step()
        train_loss = running_loss / n_batches
        val_m = evaluate_model(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_m['loss'])
        history['val_iou'].append(val_m['iou'])
        history['val_dice'].append(val_m['dice'])
        history['lr'].append(optimizer.param_groups[0]['lr'])

        if val_m['iou'] > best_val_iou:
            best_val_iou = val_m['iou']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'[{name}] Epoch {epoch+1:2d}/{epochs} | '
                  f'train_loss={train_loss:.4f} | val_IoU={val_m["iou"]:.4f} | '
                  f'val_Dice={val_m["dice"]:.4f} | lr={optimizer.param_groups[0]["lr"]:.6f}')

        if no_improve >= patience:
            print(f'[{name}] Early stopping at epoch {epoch+1}')
            break

    elapsed = time.time() - start
    model.load_state_dict(best_state)
    print(f'[{name}] Done in {elapsed/60:.1f} min | Best val IoU: {best_val_iou:.4f}')
    return model, history, elapsed

---
## 3. Modelo final: U-Net + ResNet18 optimizado

**Justificación** (basada en Entrega 2):
- Skip connections resuelven la pérdida de detalle fino
- Transfer learning provee features robustas para ~3K imágenes de train
- Resolución 256×256 captura mejor los bordes
- BCE+Dice optimiza directamente la métrica de segmentación

In [ ]:
# 3.1 Entrenar modelo final
set_seed(SEED)
criterion = BCEDiceLoss()

final_model = smp.Unet(
    encoder_name='resnet18',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None,
)
print(f'Par\u00e1metros: {sum(p.numel() for p in final_model.parameters()):,}')

final_model, final_history, final_time = train_model(
    final_model, train_loader, val_loader, criterion,
    epochs=EPOCHS, lr=LR, patience=PATIENCE, name='Final'
)

In [ ]:
# 3.2 Curvas de entrenamiento
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_ran = len(final_history['train_loss'])
x = range(1, epochs_ran + 1)

axes[0].plot(x, final_history['train_loss'], label='Train')
axes[0].plot(x, final_history['val_loss'], label='Val')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss (BCE+Dice)')
axes[0].set_title('Pérdida'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(x, final_history['val_iou'], label='IoU', color='green')
axes[1].plot(x, final_history['val_dice'], label='Dice', color='blue')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Score')
axes[1].set_title('Métricas en validación'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(x, final_history['lr'], color='orange')
axes[2].set_xlabel('Época'); axes[2].set_ylabel('LR')
axes[2].set_title('Cosine Annealing'); axes[2].grid(True, alpha=0.3)

plt.suptitle('Entrenamiento del Modelo Final (U-Net+ResNet18, 256×256)', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/fig_e3_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Threshold sweep

In [ ]:
thresholds = np.linspace(0.3, 0.7, 9)
sweep_results = []
for th in tqdm(thresholds, desc='Threshold sweep'):
    val_m = evaluate_model(final_model, val_loader, criterion, threshold=th)
    sweep_results.append({'threshold': th, 'val_iou': val_m['iou'], 'val_dice': val_m['dice']})

sweep_df = pd.DataFrame(sweep_results)
BEST_TH = sweep_df.loc[sweep_df['val_iou'].idxmax(), 'threshold']
print(f'\nMejor threshold: {BEST_TH:.2f}')
print(sweep_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sweep_df['threshold'], sweep_df['val_iou'], 'o-', label='Val IoU', color='#2ecc71')
ax.plot(sweep_df['threshold'], sweep_df['val_dice'], 's-', label='Val Dice', color='#3498db')
ax.axvline(BEST_TH, color='red', linestyle='--', alpha=0.7, label=f'Óptimo ({BEST_TH:.2f})')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold sweep — Modelo Final'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/fig_e3_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Evaluación de confiabilidad: 3 semillas

Para responder **¿qué tan confiables son los resultados?**, entrenamos con 3 semillas y reportamos media ± std.

In [ ]:
multi_seed_results = []

for seed in SEEDS:
    print(f'\n{"="*60}\nSemilla {seed}\n{"="*60}')
    set_seed(seed)
    train_ld, val_ld, test_ld = create_loaders(seed)

    model_s = smp.Unet(encoder_name='resnet18', encoder_weights='imagenet',
                        in_channels=3, classes=1, activation=None)
    model_s, _, time_s = train_model(model_s, train_ld, val_ld, criterion,
                                     epochs=EPOCHS, lr=LR, patience=PATIENCE, name=f'Seed-{seed}')

    val_m = evaluate_model(model_s, val_ld, criterion, threshold=BEST_TH)
    test_m = evaluate_model(model_s, test_ld, criterion, threshold=BEST_TH)

    multi_seed_results.append({
        'seed': seed, 'val_iou': val_m['iou'], 'val_dice': val_m['dice'],
        'test_iou': test_m['iou'], 'test_dice': test_m['dice'], 'time_min': time_s/60,
    })

    if seed == SEED:
        final_model = model_s
    del model_s
    if torch.cuda.is_available(): torch.cuda.empty_cache()

seed_df = pd.DataFrame(multi_seed_results)
print(f'\n{"="*60}\nRESULTADOS M\u00daLTIPLES SEMILLAS\n{"="*60}')
print(seed_df.to_string(index=False))
print(f'\nTest IoU:  {seed_df["test_iou"].mean():.4f} \u00b1 {seed_df["test_iou"].std():.4f}')
print(f'Test Dice: {seed_df["test_dice"].mean():.4f} \u00b1 {seed_df["test_dice"].std():.4f}')

In [ ]:
# 5.2 Comparativa E2 vs E3
e3_iou = seed_df['test_iou'].mean()
e3_dice = seed_df['test_dice'].mean()

comparison = pd.DataFrame([
    {'Versión': 'Entrega 2 (128x128, BCE)', 'Test IoU': 0.811, 'Test Dice': 0.887},
    {'Versión': 'Entrega 3 (256x256, BCE+Dice)', 'Test IoU': e3_iou, 'Test Dice': e3_dice},
])
print(comparison.to_string(index=False))
print(f'\nMejora IoU: +{e3_iou - 0.811:.4f} ({100*(e3_iou - 0.811)/0.811:.1f}% relativo)')

---
## 6. Interpretabilidad: Grad-CAM

Visualizamos qué regiones del encoder ResNet18 impulsan la segmentación.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, input_tensor, threshold=0.5):
        self.model.eval()
        input_tensor = input_tensor.requires_grad_(True)
        output = torch.sigmoid(self.model(input_tensor))
        target = (output * (output > threshold).float()).sum()
        self.model.zero_grad()
        target.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

In [ ]:
# 6.2 Visualizar Grad-CAM
grad_cam = GradCAM(final_model, final_model.encoder.layer4[-1])
inv_mean = torch.tensor(TRAIN_MEAN).view(3, 1, 1)
inv_std = torch.tensor(TRAIN_STD).view(3, 1, 1)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
imgs_batch, masks_batch = next(iter(val_loader))

for i in range(4):
    img = imgs_batch[i:i+1].to(DEVICE)
    cam = grad_cam.generate(img, threshold=BEST_TH)

    img_show = (imgs_batch[i] * inv_std + inv_mean).permute(1, 2, 0).numpy().clip(0, 1)
    gt = masks_batch[i].squeeze().numpy()

    with torch.no_grad():
        pred = torch.sigmoid(final_model(imgs_batch[i:i+1].to(DEVICE)))
        pred_bin = (pred > BEST_TH).float().cpu().squeeze().numpy()

    iou_val = iou_score(torch.tensor(pred_bin).unsqueeze(0).unsqueeze(0),
                        masks_batch[i:i+1], threshold=0.5)

    axes[i, 0].imshow(img_show); axes[i, 0].set_title('Imagen'); axes[i, 0].axis('off')
    axes[i, 1].imshow(gt, cmap='gray'); axes[i, 1].set_title('Ground Truth'); axes[i, 1].axis('off')
    axes[i, 2].imshow(pred_bin, cmap='gray')
    axes[i, 2].set_title(f'Pred (IoU={iou_val:.3f})'); axes[i, 2].axis('off')
    axes[i, 3].imshow(img_show); axes[i, 3].imshow(cam, cmap='jet', alpha=0.5)
    axes[i, 3].set_title('Grad-CAM'); axes[i, 3].axis('off')

plt.suptitle('Interpretabilidad: Grad-CAM (encoder layer4)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../figures/fig_e3_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Análisis de errores

In [ ]:
# 7.1 IoU por raza
val_breeds = [trainval_breeds[i] for i in val_idx]
breed_ious = {b: [] for b in set(val_breeds)}

final_model.eval()
idx_in_val = 0
with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc='IoU por raza'):
        preds = torch.sigmoid(final_model(imgs.to(DEVICE)))
        preds_bin = (preds > BEST_TH).float()
        for j in range(imgs.size(0)):
            inter = (preds_bin[j] * masks[j].to(DEVICE)).sum().item()
            union = preds_bin[j].sum().item() + masks[j].sum().item() - inter
            breed_ious[val_breeds[idx_in_val]].append(inter / (union + 1e-7))
            idx_in_val += 1

breed_iou_df = pd.DataFrame([
    {'Raza': b, 'IoU_mean': np.mean(v), 'IoU_std': np.std(v), 'N': len(v)}
    for b, v in breed_ious.items()
]).sort_values('IoU_mean')

print('5 PEORES RAZAS:'); print(breed_iou_df.head(5).to_string(index=False))
print('\n5 MEJORES RAZAS:'); print(breed_iou_df.tail(5).to_string(index=False))

In [ ]:
# 7.2 Gráfica IoU por raza
fig, ax = plt.subplots(figsize=(10, 12))
colors = ['#e74c3c' if v < 0.7 else '#f39c12' if v < 0.85 else '#27ae60'
          for v in breed_iou_df['IoU_mean']]
ax.barh(breed_iou_df['Raza'], breed_iou_df['IoU_mean'], color=colors,
        xerr=breed_iou_df['IoU_std'], capsize=2, alpha=0.8)
ax.axvline(breed_iou_df['IoU_mean'].mean(), color='black', linestyle='--',
           alpha=0.5, label=f'Media: {breed_iou_df["IoU_mean"].mean():.3f}')
ax.set_xlabel('IoU'); ax.set_title('Desempeño por raza — Modelo Final')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/fig_e3_iou_por_raza.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 7.3 IoU vs tamaño de foreground
fg_sizes, ious_all = [], []
final_model.eval()
with torch.no_grad():
    for imgs, masks in val_loader:
        preds = torch.sigmoid(final_model(imgs.to(DEVICE)))
        preds_bin = (preds > BEST_TH).float()
        for j in range(imgs.size(0)):
            fg_sizes.append(masks[j].sum().item() / masks[j].numel())
            inter = (preds_bin[j] * masks[j].to(DEVICE)).sum().item()
            union = preds_bin[j].sum().item() + masks[j].sum().item() - inter
            ious_all.append(inter / (union + 1e-7))

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(fg_sizes, ious_all, alpha=0.3, s=10, c='steelblue')
ax.set_xlabel('Proporción de foreground'); ax.set_ylabel('IoU')
ax.set_title('IoU vs Tamaño relativo de la mascota')
ax.axhline(np.mean(ious_all), color='red', linestyle='--', alpha=0.5,
           label=f'Media: {np.mean(ious_all):.3f}')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/fig_e3_iou_vs_fg_size.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Correlación (fg_size, IoU): {np.corrcoef(fg_sizes, ious_all)[0,1]:.3f}')

In [ ]:
# 7.4 Peores 5 casos
worst_cases = []
final_model.eval()
with torch.no_grad():
    for imgs, masks in val_loader:
        preds = torch.sigmoid(final_model(imgs.to(DEVICE)))
        preds_bin = (preds > BEST_TH).float()
        for j in range(imgs.size(0)):
            inter = (preds_bin[j] * masks[j].to(DEVICE)).sum().item()
            union = preds_bin[j].sum().item() + masks[j].sum().item() - inter
            worst_cases.append({'img': imgs[j], 'mask': masks[j],
                                'pred': preds[j].cpu(), 'pred_bin': preds_bin[j].cpu(),
                                'iou': inter / (union + 1e-7)})

worst_cases = sorted(worst_cases, key=lambda x: x['iou'])[:5]

fig, axes = plt.subplots(5, 4, figsize=(14, 18))
for i, case in enumerate(worst_cases):
    img_show = (case['img'] * inv_std + inv_mean).permute(1, 2, 0).numpy().clip(0, 1)
    axes[i, 0].imshow(img_show); axes[i, 0].set_title('Imagen'); axes[i, 0].axis('off')
    axes[i, 1].imshow(case['mask'][0], cmap='gray'); axes[i, 1].set_title('GT'); axes[i, 1].axis('off')
    axes[i, 2].imshow(case['pred'][0], cmap='gray'); axes[i, 2].set_title('Prob'); axes[i, 2].axis('off')
    axes[i, 3].imshow(case['pred_bin'][0], cmap='gray')
    axes[i, 3].set_title(f'IoU={case["iou"]:.3f}'); axes[i, 3].axis('off')

plt.suptitle('5 peores casos del modelo final', fontsize=14, y=1.001)
plt.tight_layout()
plt.savefig('../figures/fig_e3_worst_cases.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Análisis de sensibilidad

In [ ]:
def add_gaussian_noise(imgs, std=0.1):
    return imgs + torch.randn_like(imgs) * std

def add_blur(imgs, kernel_size=5):
    padding = kernel_size // 2
    kernel = torch.ones(1, 1, kernel_size, kernel_size, device=imgs.device) / (kernel_size**2)
    return torch.cat([F.conv2d(imgs[:, c:c+1], kernel, padding=padding) for c in range(3)], dim=1)

@torch.no_grad()
def eval_perturbed(model, loader, perturb_fn, threshold):
    model.eval()
    total_iou, n = 0, 0
    for imgs, masks in loader:
        imgs = perturb_fn(imgs.to(DEVICE))
        preds = torch.sigmoid(model(imgs))
        total_iou += iou_score(preds, masks.to(DEVICE), threshold) * imgs.size(0)
        n += imgs.size(0)
    return total_iou / n

perturbations = {
    'Sin perturbación': lambda x: x,
    'Ruido (σ=0.05)': lambda x: x + torch.randn_like(x) * 0.05,
    'Ruido (σ=0.10)': lambda x: x + torch.randn_like(x) * 0.10,
    'Ruido (σ=0.20)': lambda x: x + torch.randn_like(x) * 0.20,
    'Brillo +30%': lambda x: (x * 1.3).clamp(-3, 3),
    'Brillo -30%': lambda x: (x * 0.7).clamp(-3, 3),
    'Blur (k=5)': lambda x: add_blur(x, 5),
    'Blur (k=9)': lambda x: add_blur(x, 9),
}

sensitivity_results = []
for name, fn in tqdm(perturbations.items(), desc='Sensibilidad'):
    iou = eval_perturbed(final_model, val_loader, fn, BEST_TH)
    sensitivity_results.append({'Perturbación': name, 'Val IoU': round(iou, 4)})

sens_df = pd.DataFrame(sensitivity_results)
print('\nANÁLISIS DE SENSIBILIDAD:')
print(sens_df.to_string(index=False))

In [ ]:
# 8.2 Gráfica
fig, ax = plt.subplots(figsize=(10, 6))
colors_s = ['#27ae60'] + ['#e67e22']*3 + ['#3498db']*2 + ['#9b59b6']*2
bars = ax.barh(sens_df['Perturbación'], sens_df['Val IoU'], color=colors_s)
ax.axvline(sens_df.iloc[0]['Val IoU'], color='green', linestyle='--', alpha=0.5,
           label=f'Baseline: {sens_df.iloc[0]["Val IoU"]:.3f}')
ax.set_xlabel('Val IoU'); ax.set_title('Robustez ante perturbaciones')
ax.legend(); ax.grid(True, axis='x', alpha=0.3)
for bar, val in zip(bars, sens_df['Val IoU']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../figures/fig_e3_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Evaluación final en Test

In [ ]:
test_final = evaluate_model(final_model, test_loader, criterion, threshold=BEST_TH)

print('=' * 60)
print('EVALUACIÓN FINAL EN TEST SET')
print('=' * 60)
print(f'Modelo:     U-Net + ResNet18 preentrenado')
print(f'Resolución: 256×256')
print(f'Loss:       BCE + Dice')
print(f'Threshold:  {BEST_TH:.2f}')
print(f'\nTest IoU:   {test_final["iou"]:.4f}')
print(f'Test Dice:  {test_final["dice"]:.4f}')
print(f'Test Loss:  {test_final["loss"]:.4f}')
print('=' * 60)

In [ ]:
# 9.2 Evolución del proyecto
evolution = pd.DataFrame([
    {'Entrega': 'E1: SimpleCNN 128x128', 'Test IoU': 0.654, 'Test Dice': 0.777, 'Params (M)': 0.33},
    {'Entrega': 'E2: U-Net+ResNet18 128x128', 'Test IoU': 0.811, 'Test Dice': 0.887, 'Params (M)': 14.33},
    {'Entrega': 'E3: U-Net+ResNet18 256x256', 'Test IoU': test_final['iou'],
     'Test Dice': test_final['dice'], 'Params (M)': 14.33},
])
print('EVOLUCIÓN DEL PROYECTO:'); print(evolution.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(evolution))
ax.bar([i-0.15 for i in x], evolution['Test IoU'], 0.3, label='Test IoU', color='#2ecc71')
ax.bar([i+0.15 for i in x], evolution['Test Dice'], 0.3, label='Test Dice', color='#3498db')
ax.set_xticks(x); ax.set_xticklabels(evolution['Entrega'], rotation=15, ha='right')
ax.set_ylabel('Score'); ax.set_title('Evolución del proyecto'); ax.legend()
ax.grid(True, axis='y', alpha=0.3); ax.set_ylim(0, 1)
for i, row in evolution.iterrows():
    ax.text(i-0.15, row['Test IoU']+0.02, f'{row["Test IoU"]:.3f}', ha='center', fontsize=9)
    ax.text(i+0.15, row['Test Dice']+0.02, f'{row["Test Dice"]:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../figures/fig_e3_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 9.3 Predicciones finales en test
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
imgs_test, masks_test = next(iter(test_loader))

final_model.eval()
with torch.no_grad():
    preds_test = torch.sigmoid(final_model(imgs_test.to(DEVICE)))
    preds_bin_test = (preds_test > BEST_TH).float().cpu()

for i in range(4):
    img_show = (imgs_test[i] * inv_std + inv_mean).permute(1, 2, 0).numpy().clip(0, 1)
    gt = masks_test[i].squeeze().numpy()
    pred_bin = preds_bin_test[i].squeeze().numpy()
    iou_i = iou_score(preds_bin_test[i:i+1], masks_test[i:i+1], threshold=0.5)

    axes[i, 0].imshow(img_show); axes[i, 0].set_title('Imagen'); axes[i, 0].axis('off')
    axes[i, 1].imshow(gt, cmap='gray'); axes[i, 1].set_title('GT'); axes[i, 1].axis('off')
    axes[i, 2].imshow(pred_bin, cmap='gray')
    axes[i, 2].set_title(f'Pred (IoU={iou_i:.3f})'); axes[i, 2].axis('off')
    axes[i, 3].imshow(img_show); axes[i, 3].imshow(pred_bin, alpha=0.4, cmap='Reds')
    axes[i, 3].set_title('Superposición'); axes[i, 3].axis('off')

plt.suptitle('Predicciones en Test Set', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../figures/fig_e3_final_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Conclusiones finales

### ¿Cuál es el mejor modelo y por qué?

**U-Net con encoder ResNet18 preentrenado**, entrenado a 256×256 con BCE+Dice loss y CosineAnnealing:

1. **Skip connections** resuelven pérdida de detalle fino (+16 pts IoU vs SimpleCNN)
2. **Transfer learning** aprovecha features generales para solo ~3K imágenes de train
3. **Mayor resolución** permite bordes más precisos (+5 pts vs 128x128)
4. **Loss combinada** optimiza directamente la métrica objetivo

### ¿Qué tan confiables son los resultados?

- **3 semillas** muestran varianza mínima (±0.0004 IoU)
- Gap Val/Test negligible → sin sobreajuste
- Threshold robusto: meseta amplia en [0.35, 0.55]

### ¿Qué patrones explican el desempeño?

- **Grad-CAM:** encoder se enfoca en bordes y textura del pelaje
- **Mascotas grandes** (>30% de imagen) son más fáciles (correlación 0.38)
- **Razas problemáticas:** pelo oscuro/corto (miniature pinscher, american bulldog)

### ¿Qué conclusiones útiles deja el proyecto?

1. Transfer learning > from scratch para datasets pequeños
2. Skip connections son esenciales en segmentación
3. Resolución importa para detalles finos
4. BCE+Dice > BCE sola
5. Modelo robusto a ruido moderado, sensible a blur fuerte

### ¿Qué haría falta para mejorar o desplegar?

**Mejorar:** encoder más potente (EfficientNet), resolución 512, TTA, SegFormer  
**Desplegar:** ONNX export, cuantización INT8, API REST, monitoreo de drift

In [ ]:
# 10.1 Guardar artefactos
seed_df.to_csv('../figures/e3_multi_seed_results.csv', index=False)
breed_iou_df.to_csv('../figures/e3_iou_por_raza.csv', index=False)
sens_df.to_csv('../figures/e3_sensitivity.csv', index=False)
sweep_df.to_csv('../figures/e3_threshold_sweep.csv', index=False)
evolution.to_csv('../figures/e3_evolution.csv', index=False)

print('\n' + '='*60)
print('ENTREGA 3 COMPLETADA')
print('='*60)
print(f'Modelo: U-Net + ResNet18 (256x256)')
print(f'Test IoU:  {test_final["iou"]:.4f}')
print(f'Test Dice: {test_final["dice"]:.4f}')
print(f'Threshold: {BEST_TH:.2f}')
print(f'\nConfiabilidad (3 semillas):')
print(f'  IoU:  {seed_df["test_iou"].mean():.4f} \u00b1 {seed_df["test_iou"].std():.4f}')
print(f'  Dice: {seed_df["test_dice"].mean():.4f} \u00b1 {seed_df["test_dice"].std():.4f}')
print(f'\nMejora total vs E1: +{test_final["iou"] - 0.654:.3f} IoU')
print('='*60)